# EDGE-FIRE: Colab GPU with laptop data and C checkpoint storage
Start the authenticated laptop bridge first. Supply its private connection JSON at /content/edgefire_bridge_connection.json.
The preparation helper can generate a private runnable copy; never publish that private copy.
All admitted training images are used. Original ZIPs remain on the laptop; images are streamed over HTTPS.
Train transport uses aspect-preserving 512-pixel JPEG95 4:4:4; validation remains the original JPEGs.

In [ ]:
import json,hashlib,zipfile,subprocess,sys
from pathlib import Path
import requests
import torch
assert torch.cuda.is_available(), "Colab GPU required; no CPU fallback"
ROOT=Path("/content/edgefire_gpu_live")
ROOT.mkdir(exist_ok=True)
CONFIG=json.loads(Path("/content/edgefire_bridge_connection.json").read_text())
headers={"Authorization":"Bearer "+CONFIG["token"]}
print("GPU:",torch.cuda.get_device_name(0),flush=True)
response=requests.get(CONFIG["url"]+"/bootstrap.zip",headers=headers,timeout=(20,240),allow_redirects=False)
response.raise_for_status()
assert hashlib.sha256(response.content).hexdigest()==CONFIG["bootstrap_sha256"], "Bootstrap digest differs"
archive=ROOT/"bootstrap.zip"
archive.write_bytes(response.content)
del response
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        assert (ROOT/member.filename).resolve().is_relative_to(ROOT.resolve()), "Unsafe ZIP path"
    z.extractall(ROOT)
(ROOT/"connection.json").write_text(json.dumps(CONFIG))
print("Full reviewed metadata downloaded. Image ZIPs remain on the laptop.",flush=True)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.216","pyyaml","requests"],check=True)
print("GPU framework:",torch.__version__,"CUDA:",torch.cuda.is_available(),flush=True)


In [ ]:
BASE=[sys.executable,"-u",str(ROOT/"code/train_full_stream.py"),"--workspace",str(ROOT),
      "--portable-data",str(ROOT),"--bridge-config",str(ROOT/"connection.json"),
      "--weights",str(ROOT/"pretrained/yolo26n-seg.pt"),"--device","cuda:0",
      "--imgsz","512","--batch","8","--epochs","3","--threads","2"]
RUN=ROOT/"full_training"
command=BASE+["--run",str(RUN)]
if (RUN/"last.pt").is_file():
    command.append("--resume")
elif CONFIG.get("parent_remote_checkpoint"):
    parent=Path(CONFIG["parent_remote_checkpoint"])
    assert parent.is_file(), "Fork parent checkpoint missing"
    command += ["--fork-checkpoint",str(parent)]
print("Full-data GPU training; checkpoints are returned to the C: bridge folder.",flush=True)
subprocess.run(command,check=True)
